# Measure the Athlete, Not the Drill
### NFL Big Data Bowl 2027: which Combine sensor measurements survive the trip to Sundays?

This notebook reproduces every number and figure in the writeup from the raw competition files.

**Idea.** Four movement traits (top speed, burst, brake and bend) are computed with *one* pipeline (Savitzky-Golay
derivatives of x/y, acceleration split into tangential and lateral parts). That pipeline runs on every Combine drill rep
and on every regular-season NFL snap. The same trait can then be compared at the Combine and on Sundays for the same player.

**Questions.**
1. How *reliable* is each trait at the Combine (rep to rep, drill to drill) and in games (odd vs even snaps)?
2. Which Combine sources *translate* to the same trait in games, after controlling for weight and roster position, with a
   family-wise permutation test across every drill searched?
3. Where a trait translates, does it *matter* for NFL production?

Code is in the appendix cells (written to `src/bdb27/`), and the results are shown at the bottom.

In [ ]:
import glob, os, sys, subprocess
# Locate the competition files (Kaggle mounts them under /kaggle/input).
hits = glob.glob('/kaggle/input/**/players.csv', recursive=True)
if hits:
    os.environ['BDB_RAW_DIR'] = os.path.dirname(hits[0])
os.environ.setdefault('BDB_WORK_DIR', '/tmp/bdb27')  # keep data copies out of the public notebook output
print('data:', os.environ.get('BDB_RAW_DIR'), '| work dir:', os.environ['BDB_WORK_DIR'])
os.makedirs('src/bdb27', exist_ok=True); os.makedirs('scripts', exist_ok=True)
try:
    import polars
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'polars'], check=True)
sys.path.insert(0, 'src')

## Appendix A: source (written to `src/bdb27/`)

In [ ]:
%%writefile src/bdb27/__init__.py
"""Big Data Bowl 2027 analysis package."""


In [ ]:
%%writefile src/bdb27/data.py
"""Paths and loaders for the Big Data Bowl 2027 competition files.

Raw CSVs live in data/raw/ (see scripts/download_data.sh). The first call to a
loader for a large table converts it to Parquet in data/interim/ so later loads
are fast.
"""
from __future__ import annotations

import os
from pathlib import Path

import polars as pl

ROOT = Path(__file__).resolve().parents[2]
RAW = Path(os.environ.get("BDB_RAW_DIR", ROOT / "data" / "raw"))
# BDB_WORK_DIR moves the Parquet cache and processed tables elsewhere (the Kaggle notebook uses /tmp
# so that no copy of the competition data ends up in its public output).
WORK = Path(os.environ.get("BDB_WORK_DIR", ROOT / "data"))
INTERIM = WORK / "interim"
PROCESSED = WORK / "processed"
REPORTS = ROOT / "reports"

NA = ["NA", ""]
SEASONS = (2023, 2024, 2025)

# Combine position -> broad group used throughout the analysis.
POSITION_GROUPS = ("WR", "TE", "DB", "OL", "DL")


def _read(name: str, **kw) -> pl.DataFrame:
    return pl.read_csv(RAW / name, null_values=NA, infer_schema_length=100_000, **kw)


def players() -> pl.DataFrame:
    return _read("players.csv")


def combine_results() -> pl.DataFrame:
    return _read("combine_results.csv")


def career() -> pl.DataFrame:
    return _read("player_career_successes.csv")


def games() -> pl.DataFrame:
    return _read("games.csv")


def _parquet_cached(name: str) -> Path:
    INTERIM.mkdir(parents=True, exist_ok=True)
    out = INTERIM / name.replace(".csv", ".parquet")
    if not out.exists():
        pl.scan_csv(RAW / name, null_values=NA, infer_schema_length=100_000).sink_parquet(out)
    return out


def player_play() -> pl.DataFrame:
    # Empty strings are meaningful in a few columns (e.g. pass_result on runs),
    # so only "NA" is treated as null here.
    out = INTERIM / "player_play.parquet"
    if not out.exists():
        INTERIM.mkdir(parents=True, exist_ok=True)
        pl.read_csv(RAW / "player_play.csv", null_values="NA", infer_schema_length=100_000).write_parquet(out)
    return pl.read_parquet(out)


def combine_tracking() -> pl.LazyFrame:
    return pl.scan_parquet(_parquet_cached("combine_tracking.csv"))


def game_tracking(season: int | None = None) -> pl.LazyFrame:
    seasons = SEASONS if season is None else (season,)
    return pl.concat(
        [pl.scan_parquet(_parquet_cached(f"game_tracking_{s}.csv")) for s in seasons]
    )


def player_base() -> pl.DataFrame:
    """One row per drafted player: bio, draft slot, combine results, career outcomes."""
    return (
        players()
        .join(combine_results().drop("draft_year"), on="nfl_id", how="left")
        .join(career(), on="nfl_id", how="left")
        .rename({"combine_position": "pos_group"})
    )


In [ ]:
%%writefile src/bdb27/combine_features.py
"""Movement features from Combine Zebra tracking (10 Hz).

Each drill attempt becomes one row of kinematic features. Attempts are then
collapsed to one value per player and drill (the best attempt for "max"
features, the mean for the rest), giving a wide player-level table.
"""
from __future__ import annotations

import numpy as np
import polars as pl

MOVING = 1.0  # yd/s; below this, heading is too noisy to trust

# The same drill is spelled differently across Combine years.
DRILL_ALIASES = {
    "LINE": "LINE_DRILL",
    "PASS_PRO-MIRROR_DRILL": "PASS_PRO_MIRROR_DRILL",
    "OVER_THE_SHOULDER_ADJUST": "OVER_SHOULDER_ADJUST",
    "SPEED_OUT_LEFT": "SPEED_OUT_ROUTE_LEFT",
}


def _wrap(deg: np.ndarray) -> np.ndarray:
    return (deg + 180.0) % 360.0 - 180.0


def _time_to_distance(t: np.ndarray, cum: np.ndarray, s: np.ndarray, target: float, max_extrap: float = 2.0) -> float:
    """Interpolated time at which cumulative distance first reaches `target`.

    Clips sometimes end a yard or so short of the finish line; within
    `max_extrap` yards we extrapolate at the final speed.
    """
    i = int(np.searchsorted(cum, target))
    if i == 0:
        return np.nan
    if i >= len(cum):
        short = target - cum[-1]
        return float(t[-1] + short / s[-1]) if short <= max_extrap and s[-1] > 1 else np.nan
    frac = (target - cum[i - 1]) / max(cum[i] - cum[i - 1], 1e-9)
    return float(t[i - 1] + frac * (t[i] - t[i - 1]))


def attempt_features(g: pl.DataFrame) -> dict:
    t = g["t"].to_numpy()
    s = g["s"].to_numpy()
    a = g["a"].to_numpy()
    # The provided `dis` column lags and under-counts in Combine data; use the x/y path instead.
    step = np.hypot(np.diff(g["x"].to_numpy()), np.diff(g["y"].to_numpy()))
    heading = g["dir"].to_numpy()

    dt = np.diff(t)
    dt[dt <= 0] = np.nan
    accel_long = np.diff(s) / dt  # signed: + speeding up, - braking

    moving = (s[1:] > MOVING) & (s[:-1] > MOVING)
    dturn = np.abs(_wrap(np.diff(heading)))
    ang_vel = np.where(moving, dturn / dt, np.nan)

    peak_i = int(np.nanargmax(s))
    start = int(np.argmax(s > 0.5)) if (s > 0.5).any() else 0

    out = {
        "duration": float(t[-1] - t[start]),
        "distance": float(np.nansum(step[start:])),
        "peak_speed": float(np.nanmax(s)),
        "mean_speed": float(np.nanmean(s[start:])),
        "peak_accel": float(np.nanmax(a)),
        "peak_long_accel": float(np.nanmax(accel_long)) if len(accel_long) else np.nan,
        "peak_long_decel": float(-np.nanmin(accel_long)) if len(accel_long) else np.nan,
        "time_to_peak_speed": float(t[peak_i] - t[start]),
        "total_turn_deg": float(np.nansum(np.where(moving, dturn, 0.0))),
        "peak_ang_vel": float(np.nanmax(ang_vel)) if np.isfinite(ang_vel).any() else np.nan,
        # Speed held through the sharpest turn: a change-of-direction efficiency signal.
        "speed_at_peak_turn": float(s[1:][np.nanargmax(ang_vel)]) if np.isfinite(ang_vel).any() else np.nan,
    }

    if g["drill_type"][0] == "FORTY_YARD_DASH":
        # Clips begin as the athlete leaves the stance, so splits run from the first frame.
        cum = np.concatenate([[0.0], np.cumsum(step)])
        tt = t - t[0]
        for yd in (2, 5, 10, 20, 40):
            out[f"split_{yd}"] = _time_to_distance(tt, cum, s, yd)
        out["flying_10_speed"] = 10.0 / (out["split_40"] - _time_to_distance(tt, cum, s, 30))
    return out


def per_attempt(ct: pl.LazyFrame) -> pl.DataFrame:
    df = (
        ct.filter(pl.col("entity_type") == "PLAYER")
        .with_columns(
            pl.col("time").str.to_datetime(),
            pl.col("drill_name").replace(DRILL_ALIASES),
        )
        .sort("nfl_id", "drill_name", "attempt", "time")
        .with_columns(
            ((pl.col("time") - pl.col("time").first()).dt.total_milliseconds() / 1000.0)
            .over("nfl_id", "drill_name", "attempt", "event_id")
            .alias("t")
        )
        .collect()
    )
    keys = ["nfl_id", "drill_type", "drill_name", "attempt", "event_id"]
    rows = []
    for key, g in df.group_by(keys, maintain_order=True):
        if g.height < 5:
            continue
        rows.append(dict(zip(keys, key)) | attempt_features(g))
    return pl.DataFrame(rows).fill_nan(None)


# Higher is better for these, so the best attempt is the max; everything else is averaged.
_BEST_MAX = ["peak_speed", "peak_accel", "peak_long_accel", "peak_long_decel", "peak_ang_vel", "flying_10_speed"]
_BEST_MIN = ["split_2", "split_5", "split_10", "split_20", "split_40"]


def per_player_drill(att: pl.DataFrame) -> pl.DataFrame:
    feats = [c for c in att.columns if c not in {"nfl_id", "drill_type", "drill_name", "attempt", "event_id"}]
    aggs = [
        pl.col(c).max() if c in _BEST_MAX else pl.col(c).min() if c in _BEST_MIN else pl.col(c).mean()
        for c in feats
    ]
    return att.group_by("nfl_id", "drill_type", "drill_name").agg(pl.len().alias("n_attempts"), *aggs)


def wide(ppd: pl.DataFrame, min_players: int = 30) -> pl.DataFrame:
    """Pivot to one row per player with `<drill>__<feature>` columns, dropping rare drills."""
    common = ppd.group_by("drill_name").agg(pl.col("nfl_id").n_unique().alias("n")).filter(pl.col("n") >= min_players)
    ppd = ppd.join(common.select("drill_name"), on="drill_name")
    feats = [c for c in ppd.columns if c not in {"nfl_id", "drill_type", "drill_name"}]
    long = ppd.unpivot(index=["nfl_id", "drill_name"], on=feats).drop_nulls("value")
    long = long.with_columns((pl.col("drill_name") + "__" + pl.col("variable")).alias("col"))
    return long.pivot(on="col", index="nfl_id", values="value")


In [ ]:
%%writefile src/bdb27/game_features.py
"""Regular-season game performance per player (and per player-season).

Two sources:
  * player_play.csv  - position-specific production / efficiency rates
  * game_tracking_*  - in-game movement ceilings (how fast a player actually plays)
"""
from __future__ import annotations

import polars as pl

MAX_PLAUSIBLE_SPEED = 12.5  # yd/s; above this is almost always a tracking glitch

c = pl.col


def _rate(num: pl.Expr, den: pl.Expr) -> pl.Expr:
    return pl.when(den > 0).then(num / den)


def play_outcomes(pp: pl.DataFrame, games: pl.DataFrame, by_season: bool = False) -> pl.DataFrame:
    df = (
        pp.join(games.select("game_id", "season", "season_type"), on="game_id")
        .filter((c("season_type") == "REG") & (c("play_nullified_by_penalty") == "N"))
        .with_columns(
            route=c("route_ran").is_not_null(),
            pass_block=c("pass_rushers_encountered").is_not_null(),
            pass_rush=c("unblocked_pressure").is_not_null(),
            pressure_made=c("time_to_pressure").is_not_null(),
            coverage=c("coverage_assignment").is_not_null() & (c("coverage_assignment") != "NO_ASSIGNMENT"),
            yac_oe=c("yards_after_catch") - c("expected_yards_after_catch"),
        )
    )
    keys = ["nfl_id", "season"] if by_season else ["nfl_id"]
    out = df.group_by(keys).agg(
        snaps=pl.len(),
        games=c("game_id").n_unique(),
        # Receivers
        routes=c("route").sum(),
        targets=c("target").sum(),
        rec_yards=c("rec_yards").sum(),
        mean_separation=c("separation_at_pass_forward").mean(),
        mean_yac_oe=c("yac_oe").mean(),
        # Pass blockers
        pass_block_snaps=c("pass_block").sum(),
        pressures_allowed=c("pressure_allowed").sum(),
        sacks_allowed=c("sack_allowed").sum(),
        mean_peak_pressure_prob_allowed=c("peak_pressure_probability_allowed").mean(),
        # Pass rushers
        pass_rush_snaps=c("pass_rush").sum(),
        pressures=c("pressure_made").sum(),
        sacks=c("sack").sum(),
        mean_get_off=c("player_get_off").mean(),
        mean_time_to_pressure=c("time_to_pressure").mean(),
        # Coverage / run defense
        coverage_snaps=c("coverage").sum(),
        mean_cushion=c("cushion").mean(),
        tackles=c("tackle").sum(),
        tfl=c("tackle_for_loss").sum(),
    )
    return out.with_columns(
        target_rate=_rate(c("targets"), c("routes")),
        yards_per_route=_rate(c("rec_yards"), c("routes")),
        pressure_allowed_rate=_rate(c("pressures_allowed"), c("pass_block_snaps")),
        pressure_rate=_rate(c("pressures"), c("pass_rush_snaps")),
        snaps_per_game=_rate(c("snaps"), c("games")),
    ).sort(keys)


def movement_ceilings(gt: pl.LazyFrame, games: pl.DataFrame, by_season: bool = False) -> pl.DataFrame:
    keys = ["nfl_id", "season"] if by_season else ["nfl_id"]
    return (
        gt.join(games.lazy().select("game_id", "season", "season_type"), on="game_id")
        .filter((c("season_type") == "REG") & (c("s") <= MAX_PLAUSIBLE_SPEED))
        .group_by(keys)
        .agg(
            game_frames=pl.len(),
            game_top_speed=c("s").quantile(0.999),
            game_p95_speed=c("s").quantile(0.95),
            game_p99_accel=c("a").quantile(0.99),
        )
        .sort(keys)
        .collect()
    )


In [ ]:
%%writefile src/bdb27/traits.py
"""Four movement traits, measured the same way at the Combine and in NFL games.

    top_speed  max speed                                   (yd/s)
    burst      max tangential acceleration (speeding up)   (yd/s^2)
    brake      max tangential deceleration (slowing down)  (yd/s^2)
    bend       max lateral (centripetal) acceleration at speed >= BEND_MIN_SPEED (yd/s^2)

Velocity and acceleration come from Savitzky-Golay derivatives of x/y, never
from the provided s/a/dis columns, so both sources share one pipeline.

Derivatives are computed over each player's concatenated frames (one
vectorised pass) and frames within EDGE frames of a segment boundary are
discarded, which is equivalent to filtering every segment separately.
"""
from __future__ import annotations

import numpy as np
import polars as pl
from scipy.signal import savgol_filter

DT = 0.1
WINDOW, POLY = 7, 2
EDGE = WINDOW // 2
BEND_MIN_SPEED = 2.0
TRAITS = ("top_speed", "burst", "brake", "bend")
GAME_WINDOW_S = 5.0  # seconds after the snap


def _derivatives(df: pl.DataFrame, seg_cols: list[str]) -> pl.DataFrame:
    """Add v, a_tan, a_lat to frames sorted by segment then time."""
    x, y = df["x"].to_numpy(), df["y"].to_numpy()
    vx = savgol_filter(x, WINDOW, POLY, deriv=1, delta=DT)
    vy = savgol_filter(y, WINDOW, POLY, deriv=1, delta=DT)
    ax = savgol_filter(x, WINDOW, POLY, deriv=2, delta=DT)
    ay = savgol_filter(y, WINDOW, POLY, deriv=2, delta=DT)
    v = np.hypot(vx, vy)
    with np.errstate(invalid="ignore", divide="ignore"):
        a_tan = (ax * vx + ay * vy) / v
        a_lat = np.abs(ax * vy - ay * vx) / v
    return df.with_columns(v=v, a_tan=a_tan, a_lat=a_lat).with_columns(
        _i=pl.int_range(pl.len()).over(seg_cols), _n=pl.len().over(seg_cols)
    ).filter((pl.col("_i") >= EDGE) & (pl.col("_i") < pl.col("_n") - EDGE)).drop("_i", "_n")


def _segment_maxima(df: pl.DataFrame, seg_cols: list[str]) -> pl.DataFrame:
    return df.group_by(seg_cols).agg(
        top_speed=pl.col("v").max(),
        burst=pl.col("a_tan").max(),
        brake=(-pl.col("a_tan")).max(),
        bend=pl.col("a_lat").filter(pl.col("v") >= BEND_MIN_SPEED).max(),
        frames=pl.len(),
    )


def combine_attempt_traits(ct: pl.LazyFrame) -> pl.DataFrame:
    """Trait maxima for every Combine drill attempt (drill names canonicalised)."""
    from .combine_features import DRILL_ALIASES

    seg = ["nfl_id", "drill_name", "attempt", "event_id"]
    df = (
        ct.filter(pl.col("entity_type") == "PLAYER")
        .with_columns(pl.col("drill_name").replace(DRILL_ALIASES))
        .sort(*seg, "time")
        .select(*seg, "drill_type", "time", "x", "y")
        .collect()
    )
    df = df.filter(pl.len().over(seg) > WINDOW)
    return _segment_maxima(_derivatives(df, seg), seg)


def combine_traits(ct: pl.LazyFrame) -> pl.DataFrame:
    """Best value of each trait per player and drill."""
    return combine_attempt_traits(ct).group_by("nfl_id", "drill_name").agg(
        [pl.col(t).max() for t in TRAITS] + [pl.len().alias("attempts")]
    )


def game_play_traits(gt: pl.LazyFrame, ids: list[int] | None = None) -> pl.DataFrame:
    """Trait maxima for every player-play, using the first GAME_WINDOW_S seconds after the snap."""
    seg = ["game_id", "play_id", "nfl_id"]
    lf = gt if ids is None else gt.filter(pl.col("nfl_id").is_in(ids))
    df = (
        lf.with_columns(pl.col("time").str.to_datetime())
        .with_columns(snap=pl.col("time").filter(pl.col("event") == "ball_snap").first().over(seg))
        .filter(pl.col("snap").is_not_null())
        .filter((pl.col("time") >= pl.col("snap"))
                & (pl.col("time") <= pl.col("snap") + pl.duration(milliseconds=int(GAME_WINDOW_S * 1000))))
        .sort(*seg, "time")
        .select(*seg, "x", "y")
        .collect()
    )
    df = df.filter(pl.len().over(seg) > WINDOW)
    return _segment_maxima(_derivatives(df, seg), seg)


def game_player_traits(play_traits: pl.DataFrame, q: float = 0.9, min_plays: int = 50) -> pl.DataFrame:
    """A player's in-game trait = the q-quantile of his play-level maxima (his 'top 10% of snaps')."""
    return (
        play_traits.group_by("nfl_id")
        .agg([pl.col(t).quantile(q).alias(f"game_{t}") for t in TRAITS] + [pl.len().alias("game_plays")])
        .filter(pl.col("game_plays") >= min_plays)
    )


In [ ]:
%%writefile src/bdb27/translation.py
"""Translation map: does a trait measured in a Combine drill show up in the same player's NFL snaps?

For each position group, trait and Combine source (a tracked drill or an
official measurement) we report the Spearman correlation with the player's
in-game trait, and the partial Spearman correlation controlling for body
weight and NFL roster position (e.g. DT vs DE vs OLB), so that size and role
cannot masquerade as translation.
"""
from __future__ import annotations

import numpy as np
import polars as pl
from scipy.stats import rankdata

from . import traits

# Official tests where lower is better are negated so that + always means "more athletic".
OFFICIAL = {"forty": -1, "ten_yd_split": -1, "three_cone": -1, "short_shuttle": -1, "vertical": 1, "broad_jump": 1}
MIN_N = 25


def rank(a: np.ndarray) -> np.ndarray:
    """Average ranks (ties share a rank), so results never depend on row order."""
    return rankdata(a).astype(float)


def partial_spearman(x: np.ndarray, y: np.ndarray, Z: np.ndarray) -> float:
    rx, ry = rank(x), rank(y)
    X = np.column_stack([np.ones(len(x)), Z]) if Z.size else np.ones((len(x), 1))
    res = lambda v: v - X @ np.linalg.lstsq(X, v, rcond=None)[0]  # noqa: E731
    return float(np.corrcoef(res(rx), res(ry))[0, 1])


def _controls(d: pl.DataFrame) -> np.ndarray:
    cols = [rank(d["combine_weight"].to_numpy())]
    pos = d["nfl_position"].to_dummies()
    cols += [pos[c].to_numpy().astype(float) for c in pos.columns[1:]]
    Z = np.column_stack(cols)
    return Z[:, Z.std(0) > 0]


def _sources(base: pl.DataFrame, combine_traits: pl.DataFrame, trait: str) -> dict[str, pl.DataFrame]:
    """Every Combine source of `trait` as a (nfl_id, x) frame."""
    out = {
        drill: g.select("nfl_id", pl.col(trait).alias("x"))
        for (drill,), g in combine_traits.drop_nulls(trait).group_by("drill_name")
    }
    for col, sign in OFFICIAL.items():
        out[col] = base.select("nfl_id", (pl.col(col) * sign).alias("x")).drop_nulls()
    return out


def translation_map(base: pl.DataFrame, combine_traits: pl.DataFrame, game_traits: pl.DataFrame,
                    groups: tuple[str, ...]) -> pl.DataFrame:
    rows = []
    players = base.select("nfl_id", "pos_group", "nfl_position", "combine_weight").join(game_traits, on="nfl_id")
    for grp in groups:
        g = players.filter(pl.col("pos_group") == grp)
        for t in traits.TRAITS:
            for src, s in _sources(base, combine_traits, t).items():
                d = g.join(s, on="nfl_id").drop_nulls(["x", f"game_{t}", "combine_weight"])
                if d.height < MIN_N:
                    continue
                x, y = d["x"].to_numpy(), d[f"game_{t}"].to_numpy()
                rows.append(dict(group=grp, trait=t, source=src, official=src in OFFICIAL, n=d.height,
                                 rho=float(np.corrcoef(rank(x), rank(y))[0, 1]),
                                 partial=partial_spearman(x, y, _controls(d))))
    return pl.DataFrame(rows)


def add_fwer(tmap: pl.DataFrame, base: pl.DataFrame, combine_traits: pl.DataFrame, game_traits: pl.DataFrame,
             n_perm: int = 1000, seed: int = 0) -> pl.DataFrame:
    """Family-wise (max-|rho|) permutation p-values within each (group, trait) family.

    The in-game trait is shuffled across a group's players, every source in the
    family is re-correlated, and each observed |partial rho| is compared with
    the permutation distribution of the family maximum. This accounts for having
    searched over ~15-20 drills/tests per family.
    """
    rng = np.random.default_rng(seed)
    players = base.select("nfl_id", "pos_group", "nfl_position", "combine_weight").join(game_traits, on="nfl_id")
    out = []
    for (grp, t), fam in tmap.group_by(["group", "trait"], maintain_order=True):
        g = players.filter(pl.col("pos_group") == grp).drop_nulls([f"game_{t}", "combine_weight"]).sort("nfl_id")
        ids = g["nfl_id"].to_numpy()
        y = g[f"game_{t}"].to_numpy()
        pos = {pid: i for i, pid in enumerate(ids)}
        srcs = _sources(base, combine_traits, t)
        prepared = []
        for src in fam["source"].to_list():
            d = g.join(srcs[src], on="nfl_id").drop_nulls("x")
            idx = np.array([pos[p] for p in d["nfl_id"].to_list()])
            Z = _controls(d)
            X = np.column_stack([np.ones(d.height), Z])
            M = np.eye(d.height) - X @ np.linalg.pinv(X)
            rx = M @ rank(d["x"].to_numpy())
            prepared.append((idx, M, rx / np.linalg.norm(rx)))
        null_max = np.zeros(n_perm)
        for b in range(n_perm):
            yp = y[rng.permutation(len(y))]
            best = 0.0
            for idx, M, rxn in prepared:
                ry = M @ rank(yp[idx])
                best = max(best, abs(float(rxn @ ry) / np.linalg.norm(ry)))
            null_max[b] = best
        out.append(fam.with_columns(
            p_fwer=pl.col("partial").abs().map_elements(lambda v: float((null_max >= v).mean()), return_dtype=pl.Float64)))
    return pl.concat(out)


In [ ]:
%%writefile src/bdb27/reliability.py
"""How repeatable is each trait, at the Combine and on Sundays?

Three reliabilities, all computed on residualised ranks (body weight and NFL
roster position partialled out, exactly like the translation map):

* Combine retest      - attempt 1 vs attempt 2 of the same drill
* Combine cross-drill - the same trait in two *different* drills
* NFL split-half      - odd vs even snaps, Spearman-Brown corrected

If a trait is stable in games but not across Combine reps or drills, the
problem is the Combine measurement, not the trait.
"""
from __future__ import annotations

import itertools

import polars as pl

from . import traits
from .translation import _controls, partial_spearman

MIN_PAIRS = 20


def spearman_brown(r: float, k: float) -> float:
    """Reliability of the mean of k parallel measurements with single-measurement reliability r."""
    return k * r / (1 + (k - 1) * r)


def reps_needed(r: float, target: float = 0.8) -> float:
    if r <= 0:
        return float("inf")
    return target * (1 - r) / (r * (1 - target))


def combine_retest(attempts: pl.DataFrame, base: pl.DataFrame, groups) -> pl.DataFrame:
    at = (attempts.sort("nfl_id", "drill_name", "attempt")
          .with_columns(k=pl.int_range(pl.len()).over("nfl_id", "drill_name"))
          .filter(pl.col("k") < 2)
          .join(base, on="nfl_id"))
    rows = []
    for grp in groups:
        for (drill,), g in at.filter(pl.col("pos_group") == grp).group_by("drill_name"):
            w = g.pivot(on="k", index=["nfl_id", "combine_weight", "nfl_position"], values=list(traits.TRAITS)).drop_nulls()
            if w.height < MIN_PAIRS or "top_speed_1" not in w.columns:
                continue
            for t in traits.TRAITS:
                rows.append(dict(group=grp, drill=drill, trait=t, n=w.height,
                                 r=partial_spearman(w[f"{t}_0"].to_numpy(), w[f"{t}_1"].to_numpy(), _controls(w))))
    return pl.DataFrame(rows)


def cross_drill(combine_traits: pl.DataFrame, base: pl.DataFrame, groups, min_players: int = 30) -> pl.DataFrame:
    c = combine_traits.join(base, on="nfl_id")
    rows = []
    for grp in groups:
        cg = c.filter(pl.col("pos_group") == grp)
        drills = cg.group_by("drill_name").len().filter(pl.col("len") >= min_players)["drill_name"].sort().to_list()
        for a, b in itertools.combinations(drills, 2):
            pair = (cg.filter(pl.col("drill_name") == a).select("nfl_id", "combine_weight", "nfl_position", *traits.TRAITS)
                    .join(cg.filter(pl.col("drill_name") == b).select("nfl_id", *traits.TRAITS), on="nfl_id", suffix="_b"))
            for t in traits.TRAITS:
                d = pair.drop_nulls([t, f"{t}_b"])
                if d.height >= 25:
                    rows.append(dict(group=grp, drill_a=a, drill_b=b, trait=t, n=d.height,
                                     r=partial_spearman(d[t].to_numpy(), d[f"{t}_b"].to_numpy(), _controls(d))))
    return pl.DataFrame(rows)


def game_split_half(play_traits: pl.DataFrame, base: pl.DataFrame, groups, q: float = 0.9,
                    min_plays_per_half: int = 25) -> pl.DataFrame:
    halves = (play_traits.sort("game_id", "play_id")
              .with_columns(half=pl.int_range(pl.len()).over("nfl_id") % 2)
              .group_by("nfl_id", "half")
              .agg([pl.col(t).quantile(q) for t in traits.TRAITS] + [pl.len().alias("n")]))
    w = (halves.pivot(on="half", index="nfl_id", values=list(traits.TRAITS) + ["n"])
         .filter(pl.col("n_0") >= min_plays_per_half).join(base, on="nfl_id").drop_nulls())
    rows = []
    for grp in groups:
        g = w.filter(pl.col("pos_group") == grp)
        if g.height < MIN_PAIRS:
            continue
        for t in traits.TRAITS:
            r = partial_spearman(g[f"{t}_0"].to_numpy(), g[f"{t}_1"].to_numpy(), _controls(g))
            rows.append(dict(group=grp, trait=t, n=g.height, r_half=r, r=spearman_brown(r, 2)))
    return pl.DataFrame(rows)


def summary(retest: pl.DataFrame, cross: pl.DataFrame, game: pl.DataFrame) -> pl.DataFrame:
    """One row per (group, trait): median retest, median cross-drill agreement, NFL split-half."""
    rt = retest.group_by("group", "trait").agg(combine_retest=pl.col("r").median(), retest_drills=pl.len())
    cd = cross.group_by("group", "trait").agg(combine_cross_drill=pl.col("r").median(), cross_pairs=pl.len())
    gm = game.select("group", "trait", pl.col("r").alias("nfl_split_half"))
    return gm.join(rt, on=["group", "trait"], how="left").join(cd, on=["group", "trait"], how="left")


In [ ]:
%%writefile scripts/make_report.py
"""Compute every number and figure used in the writeup.

    python scripts/build_features.py   # once
    python scripts/make_report.py      # -> reports/figures/*.png, reports/results.json
"""
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import polars as pl  # noqa: E402
from matplotlib.colors import LinearSegmentedColormap  # noqa: E402

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from bdb27 import data, reliability as rl, traits, translation as tr  # noqa: E402

FIG = data.REPORTS / "figures"
GROUPS = ("WR", "DB", "OL", "DL")  # TE: too few with >= 50 tracked NFL snaps
SHORT = {"FORTY_YARD_DASH": "40 (tracked)", "SHORT_SHUTTLE": "Shuttle (tracked)", "THREE_CONE_DRILL": "3-cone (tracked)",
         "forty": "40 time", "ten_yd_split": "10-yd split", "three_cone": "3-cone time", "short_shuttle": "Shuttle time",
         "vertical": "Vertical", "broad_jump": "Broad jump"}


def short(src: str) -> str:
    if src in SHORT:
        return SHORT[src]
    words = src.replace("_DRILL", "").replace("_ROUTE", "").replace("_LEFT", " L").replace("_RIGHT", " R").split("_")
    return " ".join(words).title()[:16]


TRAIT_LABEL = {"top_speed": "Top speed", "burst": "Burst", "brake": "Brake", "bend": "Bend"}

# Reference palette (dataviz skill): validated all-pairs for 3 slots, light surface.
SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
DIVERGING = LinearSegmentedColormap.from_list("bdb_div", ["#d03b3b", "#f0b5ad", "#f0efec", "#9ec5f4", "#1c5cab"])

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.edgecolor": GRID, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10.5,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "lines.linewidth": 2, "legend.frameon": False,
})


def save(fig, name):
    FIG.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG / name, dpi=160, bbox_inches="tight")
    plt.close(fig)


def load():
    P = data.PROCESSED
    base = data.player_base()
    ct = pl.read_parquet(P / "combine_traits.parquet")
    at = pl.read_parquet(P / "combine_attempt_traits.parquet")
    pt = pl.read_parquet(P / "game_play_traits.parquet")
    gt = traits.game_player_traits(pt)
    go = pl.read_parquet(P / "game_outcomes.parquet")
    return base, ct, at, pt, gt, go


# --------------------------------------------------------------------------- figure 1
def fig_concept(base, pt):
    """Same pipeline, two settings: a DL's Run-the-Hoop rep and one of his NFL rushes, coloured by lateral acceleration."""
    ct = (data.combine_tracking()
          .filter((pl.col("entity_type") == "PLAYER") & (pl.col("drill_name") == "RUN_THE_HOOP_DRILL"))
          .collect().sort("nfl_id", "attempt", "time"))
    dl_ids = set(base.filter(pl.col("pos_group") == "DL")["nfl_id"].to_list())
    rushes = pt.filter(pl.col("nfl_id").is_in(list(dl_ids)))
    counts = rushes.group_by("nfl_id").len().filter(pl.col("len") > 300)
    pid = int(ct.join(counts, on="nfl_id").sort("len", descending=True)["nfl_id"][0])
    rep = ct.filter((pl.col("nfl_id") == pid) & (pl.col("attempt") == ct.filter(pl.col("nfl_id") == pid)["attempt"].min()))
    play = rushes.filter(pl.col("nfl_id") == pid).sort("bend", descending=True).row(len(rushes.filter(pl.col("nfl_id") == pid)) // 10, named=True)
    gt = (pl.scan_csv(data.RAW / f"game_tracking_{str(play['game_id'])[:4] if str(play['game_id'])[4:6] != '01' else int(str(play['game_id'])[:4]) - 1}.csv", null_values="NA")
          .filter((pl.col("game_id") == play["game_id"]) & (pl.col("play_id") == play["play_id"]) & (pl.col("nfl_id") == pid))
          .collect().with_columns(pl.col("time").str.to_datetime()).sort("time"))
    snap = gt["time"].filter(gt["event"] == "ball_snap")[0]
    gt = gt.filter((pl.col("time") >= snap) & (pl.col("time") <= snap + pl.duration(seconds=traits.GAME_WINDOW_S)))
    name = base.filter(pl.col("nfl_id") == pid)["display_name"][0]

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={"width_ratios": [1.25, 1]})
    vmax = 8.0
    cmap = LinearSegmentedColormap.from_list("seq", ["#cde2fb", "#6da7ec", "#2a78d6", "#104281"])
    for ax, df, title in [(axes[0], rep, "Combine: Run-the-Hoop drill"), (axes[1], gt, "NFL: one pass rush (first 5 s after snap)")]:
        x, y = df["x"].to_numpy(), df["y"].to_numpy()
        k = traits._derivatives(df.with_columns(seg=pl.lit(0)), ["seg"])
        xs, ys, lat = k["x"].to_numpy(), k["y"].to_numpy(), np.where(k["v"].to_numpy() >= traits.BEND_MIN_SPEED, k["a_lat"].to_numpy(), np.nan)
        ax.plot(x - x.mean(), y - y.mean(), color=GRID, lw=6, solid_capstyle="round", zorder=1)
        sc = ax.scatter(xs - x.mean(), ys - y.mean(), c=np.clip(lat, 0, vmax), cmap=cmap, vmin=0, vmax=vmax, s=28,
                        edgecolors=SURFACE, linewidths=0.6, zorder=2)
        ax.plot(x[0] - x.mean(), y[0] - y.mean(), marker="o", ms=9, color=INK, zorder=3)
        ax.annotate("start", (x[0] - x.mean(), y[0] - y.mean()), xytext=(6, 6), textcoords="offset points", color=INK2)
        ax.set_aspect("equal")
        ax.set_title(title)
        ax.set_xlabel("yards")
    cb = fig.colorbar(sc, ax=axes, fraction=0.025, pad=0.02)
    cb.set_label("lateral acceleration (yd/s², speed ≥ 2 yd/s)")
    fig.suptitle(f"One pipeline for both settings: {name}'s bend at the Combine and on Sundays", x=0.01, ha="left", fontsize=13, fontweight="bold")
    save(fig, "fig1_one_pipeline.png")
    return {"player": name}


# --------------------------------------------------------------------------- figure 2
def fig_reliability(summ: pl.DataFrame):
    fig, axes = plt.subplots(1, 4, figsize=(12, 3.6), sharey=True)
    series = [("combine_cross_drill", "Combine: same trait, different drill", AQUA, "s"),
              ("combine_retest", "Combine: same drill, 2nd attempt", ORANGE, "D"),
              ("nfl_split_half", "NFL: odd vs even snaps", BLUE, "o")]
    ypos = {g: i for i, g in enumerate(reversed(GROUPS))}
    for ax, t in zip(axes, traits.TRAITS):
        s = summ.filter(pl.col("trait") == t)
        for g in GROUPS:
            row = s.filter(pl.col("group") == g)
            if row.is_empty():
                continue
            vals = [row[c][0] for c, *_ in series]
            finite = [v for v in vals if v is not None]
            ax.plot([min(finite), max(finite)], [ypos[g]] * 2, color=GRID, lw=3, zorder=1)
            for (c, lab, col, mk), v in zip(series, vals):
                if v is not None:
                    ax.plot(v, ypos[g], marker=mk, ms=9, color=col, mec=SURFACE, mew=1.5, ls="none", zorder=3)
        ax.set_title(TRAIT_LABEL[t])
        ax.set_xlim(-0.2, 1.0)
        ax.axvline(0, color=INK2, lw=0.8)
        ax.set_yticks(list(ypos.values()), list(ypos.keys()))
        ax.set_xlabel("reliability (partial ρ)")
    handles = [plt.Line2D([], [], marker=mk, color=col, ls="none", ms=9, mec=SURFACE, label=lab) for _, lab, col, mk in series]
    fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(0.01, 1.08), ncol=3)
    fig.suptitle("Stable on Sundays, noisy at the Combine: change-of-direction numbers barely repeat and don't agree across drills", x=0.01, y=1.15,
                 ha="left", fontsize=13, fontweight="bold")
    save(fig, "fig2_reliability.png")


# --------------------------------------------------------------------------- figure 3
def fig_translation(tmap: pl.DataFrame):
    best = (tmap.filter(pl.col("group").is_in(GROUPS))
            .sort(pl.col("partial").abs(), descending=True)
            .group_by("group", "trait", "official", maintain_order=True).first())
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), sharey=True)
    for ax, official, title in [(axes[0], True, "Best stopwatch / tape-measure test"), (axes[1], False, "Best sensor-tracked drill")]:
        M = np.full((len(GROUPS), len(traits.TRAITS)), np.nan)
        for i, g in enumerate(GROUPS):
            for j, t in enumerate(traits.TRAITS):
                r = best.filter((pl.col("group") == g) & (pl.col("trait") == t) & (pl.col("official") == official))
                if r.is_empty():
                    continue
                M[i, j] = r["partial"][0]
                src = short(r["source"][0])
                star = "  ✱" if r["p_fwer"][0] < 0.05 else ""
                ink = SURFACE if abs(M[i, j]) > 0.45 else INK
                ax.text(j, i - 0.12, f"{M[i, j]:+.2f}{star}", ha="center", va="center", fontsize=11, fontweight="bold", color=ink)
                ax.text(j, i + 0.24, f"{src}\nn={r['n'][0]}", ha="center", va="center", fontsize=7.5, color=ink, linespacing=1.1)
        im = ax.imshow(M, cmap=DIVERGING, vmin=-0.7, vmax=0.7, aspect="auto")
        ax.set_xticks(range(4), [TRAIT_LABEL[t] for t in traits.TRAITS])
        ax.set_yticks(range(len(GROUPS)), GROUPS)
        ax.grid(False)
        ax.set_title(title)
        for s in ax.spines.values():
            s.set_visible(False)
        ax.set_xticks(np.arange(-0.5, 4, 1), minor=True)
        ax.set_yticks(np.arange(-0.5, len(GROUPS), 1), minor=True)
        ax.grid(which="minor", color=SURFACE, lw=3)
        ax.tick_params(which="minor", length=0)
    cb = fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02)
    cb.set_label("partial ρ with same trait in NFL games")
    fig.suptitle("Translation map: Combine trait vs. the same trait in NFL snaps", x=0.01, y=1.06, ha="left", fontsize=13, fontweight="bold")
    fig.text(0.01, 0.985, "Strongest single source per cell; partial Spearman ρ controlling for weight and roster position.  ✱ = family-wise permutation p < 0.05",
             ha="left", fontsize=9.5, color=INK2)
    save(fig, "fig3_translation_map.png")


# --------------------------------------------------------------------------- figure 4
def fig_speed(base, ct, gt):
    wr = base.filter(pl.col("pos_group") == "WR").join(gt, on="nfl_id")
    tracked = wr.join(ct.filter(pl.col("drill_name") == "FORTY_YARD_DASH").select("nfl_id", "top_speed"), on="nfl_id").drop_nulls(["top_speed", "combine_weight"])
    stop = wr.drop_nulls(["forty", "combine_weight"])
    r1 = tr.partial_spearman(tracked["top_speed"].to_numpy(), tracked["game_top_speed"].to_numpy(), tr._controls(tracked))
    r2 = tr.partial_spearman(-stop["forty"].to_numpy(), stop["game_top_speed"].to_numpy(), tr._controls(stop))
    to_mph = 3600 / 1760
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2), sharey=True)
    for ax, x, y, xl, r in [(axes[0], tracked["top_speed"] * to_mph, tracked["game_top_speed"] * to_mph, "Combine 40: tracked peak speed (mph)", r1),
                            (axes[1], stop["forty"], stop["game_top_speed"] * to_mph, "Combine 40: official time (s)", r2)]:
        ax.scatter(x, y, s=42, color=BLUE, edgecolors=SURFACE, linewidths=1.2, alpha=0.9)
        ax.set_xlabel(xl)
        ax.set_title(f"partial ρ = {r:+.2f}  (n = {len(x)})" if ax is axes[0] else f"partial ρ = {r:+.2f} on −time  (n = {len(x)})")
    axes[1].invert_xaxis()
    axes[0].set_ylabel("NFL top speed, 90th pct of snaps (mph)")
    fig.suptitle("WR speed travels, and the sensor reads it better than the stopwatch", x=0.01, y=1.03, ha="left", fontsize=13, fontweight="bold")
    fig.text(0.01, 0.955, "ρ = partial Spearman controlling for body weight", ha="left", fontsize=9.5, color=INK2)
    save(fig, "fig4_wr_speed.png")
    return {"wr_speed_tracked": r1, "wr_speed_stopwatch": r2, "n_tracked": tracked.height, "n_stopwatch": stop.height}


# --------------------------------------------------------------------------- figure 5
def fig_dl(base, ct, gt, go):
    dl = base.filter(pl.col("pos_group") == "DL").join(gt, on="nfl_id").join(go, on="nfl_id", how="left")
    a = dl.join(ct.filter(pl.col("drill_name") == "SHORT_SHUTTLE").select("nfl_id", pl.col("burst").alias("shuttle_burst")), on="nfl_id").drop_nulls(["shuttle_burst", "combine_weight"])
    b = dl.filter(pl.col("pass_rush_snaps") >= 100).drop_nulls(["pressure_rate", "combine_weight"])
    ra = tr.partial_spearman(a["shuttle_burst"].to_numpy(), a["game_burst"].to_numpy(), tr._controls(a))
    rb = tr.partial_spearman(b["game_burst"].to_numpy(), b["pressure_rate"].to_numpy(), tr._controls(b))
    rc = tr.partial_spearman(b["game_burst"].to_numpy(), -b["mean_get_off"].to_numpy(), tr._controls(b.drop_nulls("mean_get_off")))
    def within(d, x, y):
        out = {}
        for role, mask in (("edge", pl.col("nfl_position").is_in(["DE", "OLB"])), ("interior", ~pl.col("nfl_position").is_in(["DE", "OLB"]))):
            g = d.filter(mask)
            out[role] = (tr.partial_spearman(g[x].to_numpy(), g[y].to_numpy(), tr._controls(g)), g.height)
        return out

    wa, wb = within(a, "shuttle_burst", "game_burst"), within(b, "game_burst", "pressure_rate")
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.4))
    edge = lambda d: d["nfl_position"].is_in(["DE", "OLB"]).to_numpy()  # noqa: E731
    for ax, d, x, y, xl, yl, r, w in [
        (axes[0], a, "shuttle_burst", "game_burst", "Combine short shuttle: tracked burst (yd/s²)", "NFL burst, 90th pct of snaps (yd/s²)", ra, wa),
        (axes[1], b, "game_burst", "pressure_rate", "NFL burst, 90th pct of snaps (yd/s²)", "pressure rate per pass rush", rb, wb),
    ]:
        e = edge(d)
        ax.scatter(d[x].to_numpy()[e], d[y].to_numpy()[e], s=44, color=BLUE, marker="o", edgecolors=SURFACE, linewidths=1.2,
                   label=f"edge (DE/OLB): ρ = {w['edge'][0]:+.2f}, n = {w['edge'][1]}")
        ax.scatter(d[x].to_numpy()[~e], d[y].to_numpy()[~e], s=44, color=ORANGE, marker="D", edgecolors=SURFACE, linewidths=1.2,
                   label=f"interior (DT/NT): ρ = {w['interior'][0]:+.2f}, n = {w['interior'][1]}")
        ax.set_xlabel(xl)
        ax.set_ylabel(yl)
        ax.set_title(f"all: partial ρ = {r:+.2f} (n = {d.height})")
        ax.legend(loc="lower right" if ax is axes[0] else "upper left", fontsize=9)
    fig.suptitle("DL burst: a timed agility rep predicts NFL burst; NFL burst predicts pressure for edge rushers",
                 x=0.01, y=1.03, ha="left", fontsize=13, fontweight="bold")
    fig.text(0.01, 0.955, "ρ = partial Spearman controlling for weight and roster position, overall and within role", ha="left", fontsize=9.5, color=INK2)
    save(fig, "fig5_dl_burst.png")
    return {"dl_shuttle_to_game_burst": ra, "n_shuttle": a.height, "dl_game_burst_to_pressure": rb, "n_pressure": b.height,
            "dl_game_burst_to_getoff": rc, "within_shuttle": wa, "within_pressure": wb}


def bootstrap_ci(x, y, Z, B=2000, seed=0):
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(B):
        i = rng.integers(0, len(x), len(x))
        Zi = Z[i][:, Z[i].std(0) > 0] if Z.size else Z
        out.append(tr.partial_spearman(x[i], y[i], Zi))
    return [float(v) for v in np.nanpercentile(out, [2.5, 97.5])]


def main():
    base, ct, at, pt, gt, go = load()
    FIG.mkdir(parents=True, exist_ok=True)
    ctl = base.select("nfl_id", "pos_group", "nfl_position", "combine_weight").drop_nulls()
    res = {}

    retest = rl.combine_retest(at, ctl, GROUPS)
    cross = rl.cross_drill(ct, ctl, GROUPS)
    game = rl.game_split_half(pt, ctl, GROUPS)
    summ = rl.summary(retest, cross, game)
    summ.write_csv(data.REPORTS / "reliability_summary.csv")
    retest.write_csv(data.REPORTS / "combine_retest.csv")
    res["reliability"] = summ.sort("group", "trait").to_dicts()
    res["retest_by_drill"] = retest.to_dicts()

    tmap = tr.add_fwer(tr.translation_map(base, ct, gt, GROUPS), base, ct, gt)
    tmap.write_csv(data.REPORTS / "translation_map.csv")
    res["translation_survivors"] = tmap.filter(pl.col("p_fwer") < 0.05).sort("p_fwer").to_dicts()
    res["n_tests"] = tmap.height
    pos_drills = tmap.filter(~pl.col("official") & ~pl.col("source").is_in(["FORTY_YARD_DASH", "SHORT_SHUTTLE", "THREE_CONE_DRILL"]))
    res["position_drill_tests"] = pos_drills.height
    res["position_drill_survivors"] = pos_drills.filter(pl.col("p_fwer") < 0.05).height

    # Pooled reps-needed table: median single-rep retest per trait across groups and drills.
    reps = (retest.group_by("trait").agg(r=pl.col("r").median(), drills=pl.col("drill").n_unique())
            .with_columns(reps_for_0_8=pl.col("r").map_elements(rl.reps_needed, return_dtype=pl.Float64)))
    res["reps_needed"] = reps.to_dicts()

    res["fig1"] = fig_concept(base, pt)
    fig_reliability(summ)
    fig_translation(tmap)
    res["fig4"] = fig_speed(base, ct, gt)
    res["fig5"] = fig_dl(base, ct, gt, go)

    # Bootstrap CIs for the headline correlations.
    wr = base.filter(pl.col("pos_group") == "WR").join(gt, on="nfl_id").join(
        ct.filter(pl.col("drill_name") == "FORTY_YARD_DASH").select("nfl_id", "top_speed"), on="nfl_id").drop_nulls(["top_speed", "combine_weight"])
    res["fig4"]["ci_tracked"] = bootstrap_ci(wr["top_speed"].to_numpy(), wr["game_top_speed"].to_numpy(), tr._controls(wr))
    dl = base.filter(pl.col("pos_group") == "DL").join(gt, on="nfl_id").join(go, on="nfl_id").filter(pl.col("pass_rush_snaps") >= 100).drop_nulls(["pressure_rate", "combine_weight"])
    res["fig5"]["ci_burst_pressure"] = bootstrap_ci(dl["game_burst"].to_numpy(), dl["pressure_rate"].to_numpy(), tr._controls(dl))

    res["coverage"] = {"players": base.height, "with_game_traits": gt.height,
                       "game_player_plays": pt.height, "combine_attempts": at.height}
    (data.REPORTS / "results.json").write_text(json.dumps(res, indent=1, default=float))
    print(summ.sort("trait", "group"))
    print(reps)
    print(json.dumps({k: res[k] for k in ("fig4", "fig5", "n_tests", "position_drill_tests", "position_drill_survivors", "coverage")}, indent=1, default=float))


if __name__ == "__main__":
    main()


## Build the trait tables

In [ ]:
import time, polars as pl
from bdb27 import data, game_features as gf, traits
t0 = time.time()
data.PROCESSED.mkdir(parents=True, exist_ok=True)
games, pp = data.games(), data.player_play()
gf.play_outcomes(pp, games).write_parquet(data.PROCESSED / 'game_outcomes.parquet')
traits.combine_attempt_traits(data.combine_tracking()).write_parquet(data.PROCESSED / 'combine_attempt_traits.parquet')
traits.combine_traits(data.combine_tracking()).write_parquet(data.PROCESSED / 'combine_traits.parquet')
reg = games.filter(pl.col('season_type') == 'REG')['game_id'].to_list()
pl.concat([traits.game_play_traits(data.game_tracking(s).filter(pl.col('game_id').is_in(reg))) for s in data.SEASONS]
          ).write_parquet(data.PROCESSED / 'game_play_traits.parquet')
print(f'built in {time.time() - t0:.0f}s')

## Compute results and figures

In [ ]:
sys.path.insert(0, 'scripts')
import importlib, make_report
importlib.reload(make_report)
make_report.main()

## Results

In [ ]:
from IPython.display import Image, display
import json
for f in ['fig1_one_pipeline', 'fig2_reliability', 'fig3_translation_map', 'fig4_wr_speed', 'fig5_dl_burst']:
    display(Image(filename=f'reports/figures/{f}.png'))

In [ ]:
res = json.load(open('reports/results.json'))
print('Reliability summary (partial rho):')
display(pl.read_csv('reports/reliability_summary.csv').sort('trait', 'group'))
print('Combine reps needed for reliability 0.8 (Spearman-Brown):')
display(pl.DataFrame(res['reps_needed']))
print('Translations surviving the family-wise permutation test:')
display(pl.DataFrame(res['translation_survivors']).select('group', 'trait', 'source', 'n', 'partial', 'p_fwer'))
print(f"{res['position_drill_survivors']} of {res['position_drill_tests']} coach-led position-drill tests survive.")